In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
       # print(os.path.join(dirname, filename))
        pass

In [ ]:
!pip install -q opencv-python-headless tqdm

In [ ]:
# ============================================================================
# 0. INSTALL & IMPORTS
# ============================================================================

import os, time, zipfile
import cv2
import numpy as np
from tqdm import tqdm

# ============================================================================
# 1. CONFIGURATION – EDIT THESE PATHS
# ============================================================================
# Folder containing the 4 class subfolders (as mounted in Kaggle)
INPUT_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-augmented'            # <-- CHANGE THIS
CLASS_FOLDERS = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']

# Output directory (everything will be saved here)
OUTPUT_DIR = '/kaggle/working/moringa_segmented'
CROPPED_DIR = os.path.join(OUTPUT_DIR, 'cropped_leaves')
MASK_DIR    = os.path.join(OUTPUT_DIR, 'masks')
OVERLAY_DIR = os.path.join(OUTPUT_DIR, 'overlays')

for d in [CROPPED_DIR, MASK_DIR, OVERLAY_DIR]:
    os.makedirs(d, exist_ok=True)

# Target size for the square cropped image
TARGET_SIZE = 256

# ============================================================================
# 2. HSV COLOUR THRESHOLDS (tune these if your lighting differs)
# ============================================================================
# Green – healthy leaf area
LOWER_GREEN  = np.array([25, 40, 40])
UPPER_GREEN  = np.array([85, 255, 255])

# Yellow – interveinal chlorosis
LOWER_YELLOW = np.array([15, 50, 50])
UPPER_YELLOW = np.array([35, 255, 255])

# Brown – spots (cercospora / bacterial combined)
LOWER_BROWN  = np.array([5, 30, 30])
UPPER_BROWN  = np.array([20, 255, 200])

# Minimum area (in pixels) to keep a colour blob
MIN_AREA = 10

# ============================================================================
# 3. HELPER FUNCTIONS
# ============================================================================
def detect_leaf_bbox(img_rgb):
    """Locate the leaf using a combined colour mask and return bbox."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    mask_g = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
    mask_y = cv2.inRange(hsv, LOWER_YELLOW, UPPER_YELLOW)
    mask_b = cv2.inRange(hsv, LOWER_BROWN, UPPER_BROWN)
    leaf_mask = cv2.bitwise_or(mask_g, mask_y)
    leaf_mask = cv2.bitwise_or(leaf_mask, mask_b)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(leaf_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = img_rgb.shape[:2]
        return 0, 0, w, h
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    return x, y, w, h

def crop_and_resize(img_rgb, target_size=TARGET_SIZE):
    """Crop around the leaf bbox and resize to a square."""
    x, y, w, h = detect_leaf_bbox(img_rgb)
    cropped = img_rgb[y:y+h, x:x+w]
    cropped = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    return cropped

def generate_colour_mask(cropped_rgb):
    """
    Create a segmentation mask from the cropped image.
    Labels: 0=background, 1=healthy green, 2=yellow, 3=brown spot.
    """
    hsv = cv2.cvtColor(cropped_rgb, cv2.COLOR_RGB2HSV)

    mask_green  = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
    mask_yellow = cv2.inRange(hsv, LOWER_YELLOW, UPPER_YELLOW)
    mask_brown  = cv2.inRange(hsv, LOWER_BROWN, UPPER_BROWN)

    # Clean up noise
    def clean(m):
        kernel = np.ones((3,3), np.uint8)
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel, iterations=1)
        return cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel, iterations=1)

    mask_green  = clean(mask_green)
    mask_yellow = clean(mask_yellow)
    mask_brown  = clean(mask_brown)

    # Assign labels (priority: brown > yellow > green)
    seg = np.zeros(cropped_rgb.shape[:2], dtype=np.uint8)
    seg[mask_green > 0]  = 1
    seg[mask_yellow > 0] = 2
    seg[mask_brown > 0]  = 3

    # Ensure background is 0 everywhere outside the leaf
    leaf_union = mask_green | mask_yellow | mask_brown
    seg[leaf_union == 0] = 0
    return seg

def overlay_mask(rgb, seg):
    """Blend colour overlay on the original image (RGB)."""
    colors = {
        0: [0, 0, 0],       # bg – black
        1: [0, 255, 0],     # healthy – green
        2: [255, 255, 0],   # yellow – yellow
        3: [165, 42, 42]    # spots – brown
    }
    overlay = np.zeros_like(rgb)
    for cls_idx, col in colors.items():
        overlay[seg == cls_idx] = col
    blended = cv2.addWeighted(rgb, 0.6, overlay, 0.4, 0)
    return blended

# ============================================================================
# 4. MAIN PROCESSING LOOP (WITH TIMER)
# ============================================================================
print("\n========== Starting pipeline ==========")
total_start = time.time()

# Collect all image paths
all_images = []
for cls in CLASS_FOLDERS:
    folder = os.path.join(INPUT_DIR, cls)
    if not os.path.isdir(folder):
        print(f"Warning: {folder} not found. Skipping class '{cls}'.")
        continue
    for fname in os.listdir(folder):
        all_images.append((cls, fname, os.path.join(folder, fname)))

print(f"Found {len(all_images)} images across {len(CLASS_FOLDERS)} classes.")

# Processing loop
for cls, fname, img_path in tqdm(all_images, desc="Processing images"):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"Could not read {img_path}, skipping.")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 1. Crop leaf
    cropped = crop_and_resize(img_rgb)

    # 2. Generate mask
    mask = generate_colour_mask(cropped)

    # 3. Create overlay
    overlay = overlay_mask(cropped, mask)

    # Save with unique filename: class__originalname
    save_name = f"{cls}__{fname}"
    cv2.imwrite(os.path.join(CROPPED_DIR, save_name),
                cv2.cvtColor(cropped, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(MASK_DIR, save_name), mask)        # grayscale label PNG
    cv2.imwrite(os.path.join(OVERLAY_DIR, save_name),
                cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

total_time = time.time() - total_start
print(f"\n========== Processing finished ==========")
print(f"Time taken: {total_time/60:.2f} minutes for {len(all_images)} images")



In [ ]:
# ============================================================================
# 5. ZIP RESULTS FOR DOWNLOAD
# ============================================================================
zip_path = '/kaggle/working/moringa_segmented.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            full = os.path.join(root, file)
            arcname = os.path.relpath(full, OUTPUT_DIR)
            zf.write(full, arcname)

print(f"Results zipped to: {zip_path}")
print("You can download the ZIP from the Kaggle Output tab.")

## Direct Image Centric Model


In [ ]:
# ============================================================================
# 0. INSTALL & IMPORTS
# ============================================================================
!pip install -q tensorflow scikit-learn matplotlib seaborn tqdm

import os, time, zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications

# Enable mixed precision to save GPU memory (safe on modern GPUs)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
CROPPED_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/cropped_leaves'   # adjust path
OUTPUT_DIR = '/kaggle/working/cnn_classification_mem_safe'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32          # reduce to 16 if OOM
EPOCHS = 50
FINE_TUNE_EPOCHS = 15
LEARNING_RATE = 1e-4
TEST_SIZE = 0.2
VAL_SIZE = 0.2

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
NUM_CLASSES = 4

# ============================================================================
# 2. COLLECT FILE PATHS & LABELS (no images loaded yet)
# ============================================================================
print("========== Collecting file paths ==========")
all_filepaths = []
all_labels = []

for fname in os.listdir(CROPPED_DIR):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
    cls = fname.split('__')[0]
    if cls not in CLASS_NAMES:
        continue
    all_filepaths.append(os.path.join(CROPPED_DIR, fname))
    all_labels.append(cls)

print(f"Found {len(all_filepaths)} images")

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)
print("Class mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

from sklearn.model_selection import train_test_split
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    all_filepaths, all_labels_enc, test_size=TEST_SIZE, random_state=42, stratify=all_labels_enc
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=VAL_SIZE, random_state=42, stratify=train_val_labels
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

train_labels_onehot = tf.keras.utils.to_categorical(train_labels, NUM_CLASSES)
val_labels_onehot = tf.keras.utils.to_categorical(val_labels, NUM_CLASSES)
test_labels_onehot = tf.keras.utils.to_categorical(test_labels, NUM_CLASSES)

# ============================================================================
# 3. TF.DATA PIPELINE (memory efficient)
# ============================================================================
def load_and_preprocess(path, label):
    """Load image, resize, and apply EfficientNet preprocessing."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)   # keep in [0,255]
    
    # ADD IT HERE:
    img = applications.efficientnet.preprocess_input(img) 
    
    return img, label

def create_dataset(paths, labels_onehot, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_onehot))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_labels_onehot, BATCH_SIZE, shuffle=True)
val_ds   = create_dataset(val_paths,   val_labels_onehot,   BATCH_SIZE, shuffle=False)
test_ds  = create_dataset(test_paths,  test_labels_onehot,  BATCH_SIZE, shuffle=False)

# ============================================================================
# 4. BUILD CNN (BULLETPROOF VERSION)
# ============================================================================
print("========== Building CNN ==========")

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name="augmentation")

base_model = applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)

# We have DELETED the Lambda layer entirely here.
# EfficientNetB0 expects inputs in [0, 255], which tf.image.decode_image 
# already provides. No extra preprocessing function is needed!

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# --- INSERT SANITY CHECK HERE ---
print("Running save sanity check...")
try:
    test_save_path = os.path.join(OUTPUT_DIR, 'sanity_check_model.keras')
    model.save(test_save_path)
    print("✅ SANITY CHECK PASSED: Model saved successfully! Safe to train.")
    os.remove(test_save_path) 
except Exception as e:
    print(f"❌ SANITY CHECK FAILED: The model cannot be saved. Do not train yet!")
    print(f"Error: {e}")
    raise SystemExit("Stopping execution to prevent wasted training time.")


In [ ]:

# ============================================================================
# 5. TRAINING WITH EARLY STOPPING & FINE-TUNING
# ============================================================================
print("========== Training ==========")
train_start = time.time()

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr  = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

train_time = time.time() - train_start
print(f"Initial training: {train_time/60:.2f} min")

# Fine-tuning
print("========== Fine‑tuning ==========")
base_model.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Merge histories
for key in history.history:
    history.history[key].extend(history_fine.history[key])

# Save model – should now work without pickling error
model.save(os.path.join(OUTPUT_DIR, 'cnn_model.keras'))


In [ ]:

# ============================================================================
# 6. PLOT TRAINING CURVES
# ============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()


In [ ]:

# ============================================================================
# 7. EVALUATE ON TEST SET
# ============================================================================
print("========== Test Evaluation ==========")
test_pred_prob = model.predict(test_ds)
test_pred = np.argmax(test_pred_prob, axis=1)

acc = accuracy_score(test_labels, test_pred)
print(f"\nTest Accuracy: {acc:.4f}")

report = classification_report(test_labels, test_pred, target_names=CLASS_NAMES)
print("\nClassification Report:\n", report)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

cm = confusion_matrix(test_labels, test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()


In [ ]:

# ============================================================================
# 8. PACKAGE RESULTS
# ============================================================================
total_time = time.time() - train_start
print(f"\n========== Pipeline Complete ==========")
print(f"Total training time: {total_time/60:.2f} min")

zip_path = '/kaggle/working/cnn_classification_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            full = os.path.join(root, file)
            arcname = os.path.relpath(full, OUTPUT_DIR)
            zf.write(full, arcname)

print(f"Results zipped to: {zip_path}")